Acá quiero ver si la inflación está siendo tomada como dato temporal o no. Es por eso que dejo foto mes como feature

In [1]:
%run 1_home.ipynb

2. La serie del IPC (INDEC, nivel general nacional, base dic-2016=100) y la función que deflacta. Mes de referencia: enero 2019.

In [2]:
# IPC nacional, INDEC via datos.gob.ar (serie 148.3_INIVELNAL_DICI_M_26)
IPC = {
    201901: 189.6101, 201902: 196.7501, 201903: 205.9571, 201904: 213.0517,
    201905: 219.5691, 201906: 225.5370, 201907: 230.4940, 201908: 239.6077,
    201909: 253.7102, 201910: 262.0661, 201911: 273.2158, 201912: 283.4442,
}
MES_REFERENCIA = 201901

def columnas_monetarias(df):
    return [c for c in df.columns
            if c.startswith("m") or c.startswith("Master_m") or c.startswith("Visa_m")]

def deflactar(df, indice, mes_referencia=MES_REFERENCIA, columnas=None):
    """monto / indice del mes * indice del mes de referencia"""
    out = df.copy()
    columnas = columnas or columnas_monetarias(df)
    factor = df.foto_mes.map(indice).astype("float32")
    out[columnas] = out[columnas].astype("float32").div(factor, axis=0) * indice[mes_referencia]
    return out

print(len(columnas_monetarias(cargar_datos().head(1))), "columnas monetarias")

73 columnas monetarias


In [3]:
df = cargar_datos()
df_ipc = deflactar(df, IPC)
datos, features = agregar_lags_deltas(df_ipc)

# Lo único que cambia contra el brazo 3: le devolvemos la referencia temporal
# que la deflación le sacó, pero explícita en vez de escondida en los niveles.
datos["mes"] = datos.foto_mes
features = features + ["mes"]

print(datos.shape, len(features), "features")

(1563565, 764) 761 features


In [4]:
import os, shutil

ESTUDIO_LOCAL  = "/home/ds/datasets/optuna_mes.db"
ESTUDIO_BUCKET = "/home/ds/buckets/b1/optuna_mes.db"

if not os.path.exists(ESTUDIO_LOCAL) and os.path.exists(ESTUDIO_BUCKET):
    shutil.copy(ESTUDIO_BUCKET, ESTUDIO_LOCAL)

estudio_mes = optimizar(datos, features, "ipc_con_mes",
                        storage=f"sqlite:///{ESTUDIO_LOCAL}")

shutil.copy(ESTUDIO_LOCAL, ESTUDIO_BUCKET)

[I 2026-11-16 02:32:08,335] A new study created in RDB with name: ipc_con_mes


[ipc_con_mes] trials completos: 0, faltan: 50


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-11-16 02:32:49,842] Trial 0 finished with value: 0.9220993871464255 and parameters: {'num_leaves': 174, 'learning_rate': 0.21119729380744556, 'min_data_in_leaf': 3555, 'feature_fraction': 0.4261986361098682, 'bagging_fraction': 0.47360212107881416}. Best is trial 0 with value: 0.9220993871464255.
[I 2026-11-16 02:33:12,738] Trial 1 finished with value: 0.9218720103695508 and parameters: {'num_leaves': 27, 'learning_rate': 0.20936112014418523, 'min_data_in_leaf': 1107, 'feature_fraction': 0.9093362456134209, 'bagging_fraction': 0.97222629698359}. Best is trial 0 with value: 0.9220993871464255.
[I 2026-11-16 02:33:50,261] Trial 2 finished with value: 0.9277577885203364 and parameters: {'num_leaves': 146, 'learning_rate': 0.11949484675078148, 'min_data_in_leaf': 7829, 'feature_fraction': 0.9995891079559249, 'bagging_fraction': 0.6947702612827709}. Best is trial 2 with value: 0.9277577885203364.
[I 2026-11-16 02:34:26,395] Trial 3 finished with value: 0.9280528976000866 and paramet

'/home/ds/buckets/b1/optuna_mes.db'

In [5]:
res_mes, curvas_mes, probas_mes = correr_semillas(datos, features, estudio_mes)
res_mes.to_csv("/home/ds/buckets/b1/mes_5semillas.csv", index=False)
res_mesres_base = pd.read_csv("/home/ds/buckets/b1/baseline_5semillas.csv")
res_ipc  = pd.read_csv("/home/ds/buckets/b1/ipc_5semillas.csv")

print("¿recupera lo que perdió al deflactar?")
comparar(res_ipc, res_mes, "IPC sin mes", "IPC con mes")

print("\n¿llega al nominal?")
comparar(res_base, res_mes, "nominal", "IPC con mes")

  semilla 100129: $287,100,000 con 4,040 envios (279s)
  semilla 100183: $286,715,000 con 4,614 envios (302s)
  semilla 100151: $289,822,500 con 3,981 envios (267s)
  semilla 500041: $287,787,500 con 4,535 envios (262s)
  semilla 900007: $286,550,000 con 4,580 envios (261s)
media $287,595,000  desvio $1,333,254
¿recupera lo que perdió al deflactar?
IPC sin mes: $286,418,000   IPC con mes: $287,595,000
diferencia media: $1,177,000  (+0.41%)
gana el experimento en 5 de 5 semillas
Wilcoxon necesita al menos 6 pares para dar un p confiable

¿llega al nominal?


NameError: name 'res_base' is not defined

In [ ]:
res_base = pd.read_csv("/home/ds/buckets/b1/baseline_5semillas.csv")
res_ipc  = pd.read_csv("/home/ds/buckets/b1/ipc_5semillas.csv")

print("¿recupera lo que perdió al deflactar?")
comparar(res_ipc, res_mes, "IPC sin mes", "IPC con mes")

print("\n¿llega al nominal?")
comparar(res_base, res_mes, "nominal", "IPC con mes")

In [ ]:
res_grupos = pd.read_csv("/home/ds/buckets/b1/grupos_5semillas.csv")
grafico_comparacion({
    "nominal":    res_base,
    "IPC único":  res_ipc,
    "por grupos": res_grupos,
    "IPC + mes":  res_mes,
}).show()